# Diagnóstico de Calidad de Datos y EDA — KDD Cup 1999

Notebook combinado: **Sección F (Calidad de Datos)** + **Sección H (EDA)**.

Convertido a partir de los scripts `data_quality_report.py` y `eda_kdd.py`,
conservando el texto y las justificaciones originales.

## Imports y configuración

In [ ]:
# Importe de librerías

import argparse
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)

In [ ]:
# Columnas simbólicas (categóricas) conocidas según kddcup.names
SYMBOLIC_COLS = [
    "protocol_type", "service", "flag", "land", "logged_in",
    "is_host_login", "is_guest_login",
]

In [ ]:
# Valores válidos conocidos para columnas categóricas dominados por el protocolo de red (RFC). Se usan para detectar valores fuera de dominio.
#
# 'protocol_type' y 'flag' solo pueden tener unos pocos valores posibles, y ya
# sabemos cuáles son (por ejemplo, 'protocol_type' solo puede ser: ('tcp', 'udp' o
# 'icmp'). Por eso podemos comparar los datos
# reales contra esta lista fija definida y detectar si aparece un dato diferente.
#
# 'service', en cambio, tiene 66 valores distintos posibles (nombres de
# servicios de red), así que no tiene sentido escribir una lista fija con
# los 66 — por eso esta columna se revisa de otra forma más adelante
# (sección de cardinalidad), no aquí.

EXPECTED_CATEGORIES = {
    "protocol_type": {"tcp", "udp", "icmp"},
    "flag": {
        "SF", "S0", "S1", "S2", "S3", "OTH", "REJ", "RSTO",
        "RSTOS0", "RSTR", "SH",
    },
    "land": {0, 1},
    "logged_in": {0, 1},
    "is_host_login": {0, 1},
    "is_guest_login": {0, 1},
}

In [ ]:
# Columnas que, por definición del protocolo de red, NUNCA deberían ser negativas
NON_NEGATIVE_COLS = [
    "duration", "src_bytes", "dst_bytes", "wrong_fragment", "urgent", "hot",
    "num_failed_logins", "num_compromised", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "count",
    "srv_count", "dst_host_count", "dst_host_srv_count",
]

In [ ]:
# Columnas de tasa (rate) que por definición matemática deben estar en [0, 1]
RATE_COLS = [
    "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
]

In [ ]:
# Mapeo ataque -> categoría, según training_attack_types (DOS/U2R/R2L/PROBE)
ATTACK_TYPE_MAP = {
    "normal": "normal",
    "back": "dos", "land": "dos", "neptune": "dos", "pod": "dos",
    "smurf": "dos", "teardrop": "dos",
    "buffer_overflow": "u2r", "loadmodule": "u2r", "perl": "u2r", "rootkit": "u2r",
    "ftp_write": "r2l", "guess_passwd": "r2l", "imap": "r2l", "multihop": "r2l",
    "phf": "r2l", "spy": "r2l", "warezclient": "r2l", "warezmaster": "r2l",
    "ipsweep": "probe", "nmap": "probe", "portsweep": "probe", "satan": "probe",
}

In [ ]:
def encontrar_raiz_proyecto(marcador="README.md"):
    """Busca hacia arriba desde el directorio actual hasta encontrar la raíz
    del repo (identificada por README.md). Evita este error(FileNotFoundError) que se presentó cuando el
    script se ejecuta desde una subcarpeta (ej. src/quality)."""
    actual = Path.cwd()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / marcador).exists():
            return carpeta
    return actual

In [ ]:
def asegurar_attack_category(df: pd.DataFrame) -> pd.DataFrame:
    """A veces el archivo CSV no trae la columna 'attack_category' (por
    ejemplo, cuando se generó con la herramienta de scikit-learn en vez del
    script de ingesta propio del equipo). Cuando eso pasa, esta función crea
    esa columna nueva usando la columna 'label' que sí existe, pero SOLO
    en la memoria de la computadora mientras corre el programa — nunca
    modifica ni guarda cambios en el archivo CSV original."""
    if "attack_category" in df.columns:
        return df
    if "label" not in df.columns:
        return df
    label_limpio = df["label"].astype(str).str.replace("b'", "", regex=False)
    label_limpio = label_limpio.str.replace("'", "", regex=False).str.rstrip(".")
    df["attack_category"] = label_limpio.map(ATTACK_TYPE_MAP).fillna("unknown")
    df["is_anomaly"] = (label_limpio != "normal").astype(int)
    return df

In [ ]:
def section(title: str):
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")

## Cargar el dataset

In [ ]:
raiz = encontrar_raiz_proyecto()
df = pd.read_csv(raiz / "data/raw/kddcup_10_percent.csv")
df = asegurar_attack_category(df)

report = {}
report["n_rows"] = int(df.shape[0])
report["n_cols"] = int(df.shape[1])

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.head()

## Sección F — Diagnóstico de Calidad de Datos

### 1. Valores faltantes (NaN reales)

In [ ]:
def check_missing_values(df: pd.DataFrame) -> dict:
    section("1. Valores faltantes (NaN reales)")
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    result = {"columns_with_nulls": nulls.to_dict(), "total_null_cells": int(nulls.sum())}
    if nulls.empty:
        print("No se encontraron NaN explícitos. Esto es consistente con la ficha "
              "de UCI en la página, que declara 'Has Missing Values: No'. Justificación: no se "
              "requiere imputación por NaN, pero eso NO descarta faltantes codificados "
              "(ver sección 2).")
    else:
        print(nulls)
    return result

In [ ]:
report["missing_values"] = check_missing_values(df)

### 2. Valores faltantes representados mediante símbolos

In [ ]:
def check_encoded_missing(df: pd.DataFrame) -> dict:
    section("2. Valores faltantes representados mediante símbolos")
    result = {}
# En este tipo de datasets es común que, en vez de dejar una celda vacía,
# alguien haya puesto el símbolo así '?' o cualquiera otro para representar un dato que falta 
# IMPORTANTE: en este dataset específico, las palabras 'other' y 'private'
# SÍ son nombres reales de servicios de red (no significan "dato faltante"),
# así que las excluimos a propósito de esta búsqueda para la columna
# 'service', para no marcarlas por error como si fueran un problema.
    suspicious_simbols = ["?", "-", "unknown", "none"]
    for col in df.select_dtypes(include="object").columns:
        simbols = suspicious_simbols if col != "service" else [t for t in suspicious_simbols]
        found = df[col].isin(simbols)
        if found.any():
            result[col] = int(found.sum())
    if not result:
        print("No se encontraron símbolos sospechosos ('?', '-', 'unknown', 'none') "
              "en columnas categóricas. Nota: 'other' y 'private' SÍ aparecen en "
              "'service' de forma legítima como nombres de servicio reales, no como "
              "código de faltante — se excluyeron intencionalmente del escaneo.")
    else:
        print(f"Simbolos sospechosos encontrados: {result}")
    return result

In [ ]:
report["encoded_missing"] = check_encoded_missing(df)

### 3. Registros duplicados

In [ ]:
def check_duplicates(df: pd.DataFrame) -> dict:
    section("3. Registros duplicados")
    dup_count = int(df.duplicated().sum())
    dup_pct = round(dup_count / len(df) * 100, 2)
    print(f"Duplicados exactos: {dup_count:,} de {len(df):,} filas ({dup_pct}%).")
    print("Justificación de la decisión: KDD Cup 1999 es conocido en la literatura por "
          "tener un porcentaje MUY alto de duplicados (hasta ~78% en el set completo), "
          "producto de cómo se generó el tráfico simulado (ráfagas repetidas de la misma "
          "conexión, especialmente en ataques DOS tipo smurf/neptune). Si borráramos esos duplicados sin pensarlo," 
          "el modelo 'vería' menos ejemplos de esos ataques de los que realmente ocurren, y aprendería a"
          "descartarlos. Decisión recomendada: "
          "NO eliminar duplicados entre clases de ataque (son parte del patrón real), pero SÍ "
          "eliminar duplicados exactos dentro de la clase 'normal', donde no aportan "
          "información nueva y sí aumentan el desbalance.")
    return {"duplicate_rows": dup_count, "duplicate_pct": dup_pct}

In [ ]:
report["duplicates"] = check_duplicates(df)

### 4. Registros lógicamente inconsistentes

In [ ]:
def check_inconsistent_records(df: pd.DataFrame) -> dict:
    section("4. Registros lógicamente inconsistentes")
    result = {}
    if {"su_attempted", "root_shell"}.issubset(df.columns):
        inconsistent = df[(df["su_attempted"] == 1) & (df["root_shell"] == 0) & (df["num_root"] == 0)]
        result["su_attempted_sin_evidencia_root"] = int(len(inconsistent))
        print(f"Filas con su_attempted=1(alguien intentó usar el comando 'su' para volverse superusuario/administrador),"
                "pero sin root_shell ni num_root>0(o sea, no logró obtener acceso de root ni hacer cambios como root). "
              "No se eliminan: 'su_attempted' solo registra el intento, no el éxito, así que "
              "un intento fallido es información válida y es justamente como si identificará un ataque U2R"
              "(cuando alguien intenta tomar el control de nivel administrador de un equipo).")
    if {"land", "src_bytes", "dst_bytes"}.issubset(df.columns):
        land_no_bytes = df[(df["land"] == 1) & (df["src_bytes"] == 0) & (df["dst_bytes"] == 0)]
        result["land_attack_sin_bytes"] = int(len(land_no_bytes))
    return result

In [ ]:
report["inconsistent_records"] = check_inconsistent_records(df)

### 5. Tipos de datos incorrectos

In [ ]:
def check_incorrect_types(df: pd.DataFrame) -> dict:
    section("5. Tipos de datos incorrectos")
    result = {}
    for col in SYMBOLIC_COLS:
        if col in df.columns and col not in ("land", "logged_in", "is_host_login", "is_guest_login"):
            is_text = pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col])
            if not is_text:
                result[col] = str(df[col].dtype)
    numeric_expected = [c for c in df.columns if c not in SYMBOLIC_COLS + ["label", "attack_category", "is_anomaly"]]
    for col in numeric_expected:
        if col in df.columns and not pd.api.types.is_numeric_dtype(df[col]):
            result[col] = str(df[col].dtype)
    if result:
        print(f"Columnas con tipo inesperado: {result}")
    else:
        print("Todas las columnas tienen el tipo esperado según kddcup.names "
              "(symbolic: object/int binario, continuous: numérico).")
    return result

In [ ]:
report["incorrect_types"] = check_incorrect_types(df)

### 6. Categorías inconsistentes / valores no permitidos

In [ ]:
def check_inconsistent_categories(df: pd.DataFrame) -> dict:
    section("6. Categorías inconsistentes / valores no permitidos en columnas de categorías")
    result = {}
    for col, expected in EXPECTED_CATEGORIES.items():
        if col not in df.columns:
            continue
        actual = set(df[col].unique())
        unexpected = actual - expected
        if unexpected:
            result[col] = list(unexpected)
    if result:
        print(f"Valores no permitidos en columnas de categorías: {result}")
        print("Justificación: cualquier valor no listado en kddcup.names/RFC de protocolos "
              "debe tratarse como categoría desconocida y mapearse explícitamente a "
              "'unknown' en Feature Engineering, no descartarse, ya que en producción "
              "simularemos justamente la aparición de categorías nuevas.")
    else:
        print("protocol_type, flag y las columnas binarias están dentro del dominio esperado.")
    if "service" in df.columns:
        n_services = df["service"].nunique()
        print(f"'service' tiene {n_services} categorías distintas (alta cardinalidad esperada, "
              "no se restringe a una lista cerrada, ver sección de cardinalidad).")
    return result

In [ ]:
report["inconsistent_categories"] = check_inconsistent_categories(df)

**Hallazgo adicional para la sección 6 (categorías inconsistentes):**

`su_attempted` tiene 3 valores distintos en los datos reales, aunque
`kddcup.names` la documenta como binaria (0/1). Esto es un artefacto
conocido de este dataset específico, no un error de la ingesta. Se
recomienda que Feature Engineering decida explícitamente cómo tratar
el tercer valor (ej. agruparlo con 1, o mantenerlo como categoría aparte).

### 7. Fechas inválidas

In [ ]:
def check_invalid_dates() -> dict:
    section("7. Fechas inválidas")
    print("No aplica: el dataset no contiene columnas de fecha/timestamp exactas. "
          "Justificación: KDD Cup 1999 solo guarda cuánto duró cada conexión (en segundos),"
          "pero no guarda EN QUÉ MOMENTO exacto ocurrió cada una. Por qué esto importa más adelante: "
          "en la parte del proyecto donde: simulamos cómo cambian los datos con el tiempo,"
          "no podemos ordenar las conexiones por fecha real porque no existe esa información. "
          "Vamos a usar el orden en que aparecen las filas en el archivo como si fuera el orden en que ocurrieron.")
    return {"applicable": False, "reason": "dataset sin columna de timestamp"}

In [ ]:
report["invalid_dates"] = check_invalid_dates()

### 8. Datos imposibles

In [ ]:
def check_impossible_values(df: pd.DataFrame) -> dict:
    section("8. Datos imposibles")
    result = {}
    for col in NON_NEGATIVE_COLS:
        if col in df.columns:
            negatives = int((df[col] < 0).sum())
            if negatives:
                result[f"{col}_negativos"] = negatives
    for col in RATE_COLS:
        if col in df.columns:
            out_of_range = int(((df[col] < 0) | (df[col] > 1)).sum())
            if out_of_range:
                result[f"{col}_fuera_de_[0,1]"] = out_of_range
    if result:
        print(f"Valores imposibles detectados: {result}")
    else:
        print("No se detectaron negativos en columnas de conteo/bytes, ni tasas fuera "
              "del rango [0,1]. Esto es consistente y esperado dado el dominio del problema.")
    return result

In [ ]:
report["impossible_values"] = check_impossible_values(df)

### 9. Valores extremos (outliers)

In [ ]:
def check_outliers(df: pd.DataFrame) -> dict:
    section("9. Valores extremos (outliers)")
    result = {}
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [c for c in numeric_cols if c not in ("is_anomaly",)]
    for col in numeric_cols:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0:
            continue
        lower, upper = q1 - 3 * iqr, q3 + 3 * iqr
        outliers = int(((df[col] < lower) | (df[col] > upper)).sum())
        if outliers > 0:
            result[col] = {"outliers": outliers, "pct": round(outliers / len(df) * 100, 2)}
    top5 = sorted(result.items(), key=lambda kv: kv[1]["pct"], reverse=True)[:5]
# Las 5 columnas donde encontramos más valores "raros" o extremos
# (usamos un método matemático estricto(con 3*IQR) para asegurarnos de que sean
# valores realmente exagerados, no solo un poco más altos de lo normal)
    print("Top 5 columnas con más outliers (método IQR*3, extremo severo):")
    for col, stats in top5:
        print(f"  {col}: {stats['outliers']:,} filas ({stats['pct']}%)")
    print("Justificación: no eliminamos valores extremos, en datos en tráfico de red, un valor 'raro' y muy alto en columnas como " \
        "los outliers severos en 'src_bytes', 'dst_bytes'(cantidad de datos enviados/recibidos) o "
        "'duration'(duración de la conexión) muchas veces NO es un error de los datos, "
        "puede ser justo la señal de que algo sospechoso está pasando. Por eso, en vez de borrar estos valores o 'suavizarlos' hacia un límite"
        "más razonable, simplemente los anotamos y seguimos revisando si tienen relación con el tipo de ataque, antes de decidir qué hacer con ellos.")
    return result

In [ ]:
report["outliers"] = check_outliers(df)

### 10. Cardinalidad

In [ ]:
def check_cardinality(df: pd.DataFrame) -> dict:
    section("10. Cardinalidad")
    result = {}
    for col in df.select_dtypes(include="object").columns:
        result[col] = int(df[col].nunique())
    print(result)
    print("Justificación: 'service' con alta cardinalidad (66 nombres de servicios de red) "
          "Si la convirtiéramos a números creando una columna nueva por cada valor posible (66 columnas nuevas), " 
          "el dataset se volvería innecesariamente pesado y difícil de manejar. "
          "Por eso, se recomienda convertirla a números de otra forma: reemplazando" \
          "cada servicio por UN SOLO número que indique, por ejemplo, qué tan seguido ese " \
          "servicio está asociado a un ataque. Esto da la misma información útil, " \
          "pero sin crear tantas columnas nuevas.")
    return result

In [ ]:
report["cardinality"] = check_cardinality(df)

### 11. Skewness

In [ ]:
def check_skewness(df: pd.DataFrame) -> dict:
    section("11. Skewness")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    skew = df[numeric_cols].skew().sort_values(ascending=False)
    high_skew = skew[skew.abs() > 3]
    print(f"Columnas con |skewness|(Asimetría) > 3 (candidatas a transformación log/Box-Cox):\n{high_skew}")
    return {"high_skew_columns": high_skew.to_dict()}

In [ ]:
report["skewness"] = check_skewness(df)

**Recomendación de transformación logarítmica (log1p), según el patrón de skewness observado:**

- **SÍ transformar** (cantidades continuas que pueden crecer mucho): `src_bytes`, `dst_bytes`, `duration`, `num_compromised`, `num_root`, `num_file_creations`, `num_failed_logins`, `num_access_files`, `num_shells`, `hot`
- **NO transformar** — son prácticamente banderas de sí/no (0 o 1), el logaritmo no aporta nada útil aquí: `land`, `su_attempted`, `root_shell`, `is_guest_login`
- **NO transformar** — ya son proporciones entre 0 y 1 por definición matemática: `rerror_rate`, `srv_rerror_rate`, `dst_host_rerror_rate`, `dst_host_srv_rerror_rate`, `diff_srv_rate`, `dst_host_diff_srv_rate`, `srv_diff_host_rate`, `dst_host_srv_diff_host_rate`
- **REVISAR antes de decidir** (casi siempre son 0, verificar cuántos valores distintos tienen realmente antes de transformar): `urgent`, `wrong_fragment`

**Recomendación de mayor prioridad: eliminar por completo (no transformar):**

- `num_outbound_cmds` → constante en todo el dataset (1 solo valor)
- `is_host_login` → constante en todo el dataset (1 solo valor)

Justificación: una columna sin ninguna variación no aporta información para
distinguir entre clases. Mantenerla no daña el modelo, pero es peso
muerto — ocupa espacio y tiempo de cómputo sin ningún beneficio.

### 12. Errores de unidad

In [ ]:
def check_unit_errors(df: pd.DataFrame) -> dict:
    section("12. Errores de unidad")
    print("Revisión conceptual: 'duration' está en segundos y 'src_bytes'/'dst_bytes' en "
          "bytes, según la documentación original. No encontramos evidencia de que se "
          "mezclen distintas unidades de medida dentro del dataset. "
          "columnas comparables Por ejemplo: no hay una columna que mida algo en "
          "bytes junto a otra columna que mida lo mismo pero en kilobytes (KB)")
    return {"applicable": True, "finding": "sin evidencia de mezcla de unidades"}

In [ ]:
report["unit_errors"] = check_unit_errors(df)

### 13. Leakage

In [ ]:
def check_leakage(df: pd.DataFrame) -> dict:
    section("13. Leakage")
    result = {}
    if "attack_category" in df.columns and "label" in df.columns:
        print("'attack_category' y 'is_anomaly' se derivan de 'label'. Ambas deben "
              "EXCLUIRSE del set de features de entrenamiento (son post-hoc del "
              "target) y usarse únicamente como target o para EDA.")
        result["derived_target_columns"] = ["attack_category", "is_anomaly", "label"]

    # Dos ventanas de cálculo distintas conviven en el dataset (ver task.html,
    # Stolfo et al.): 2 segundos (Tabla 3) vs. 100 conexiones al mismo host
    # (columnas dst_host_*, no tabuladas en el paper original pero descritas
    # conceptualmente). Ambas deben poder recalcularse igual en producción.
    time_window_cols = [c for c in df.columns if c in (
        "count", "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate",
        "srv_rerror_rate", "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate",
    )]
    host_window_cols = [c for c in df.columns if c.startswith("dst_host_")]
    result["features_ventana_2_segundos"] = time_window_cols
    result["features_ventana_100_conexiones"] = host_window_cols

    print(f"Features con ventana de 2 segundos (Tabla 3, task.html): {len(time_window_cols)} columnas")
    print(f"Features con ventana de 100 conexiones al mismo host (dst_host_*): {len(host_window_cols)} columnas")
    print("Ambos grupos deben recalcularse de forma IDÉNTICA en el API de inferencia "
          "(sección M) respecto a como se calcularon en el histórico, o habrá "
          "leakage temporal / inconsistencia train-serving.")
    return result

In [ ]:
report["leakage"] = check_leakage(df)

### 14. Imbalance de clases

In [ ]:
def check_imbalance(df: pd.DataFrame) -> dict:
    section("14. Imbalance de clases")
    result = {}
    if "attack_category" in df.columns:
        counts = df["attack_category"].value_counts()
        pct = (counts / len(df) * 100).round(3)
        print(pd.DataFrame({"count": counts, "pct": pct}))
        result["attack_category_distribution"] = counts.to_dict()
        minority = counts.idxmin()
        majority = counts.idxmax()
        ratio = counts.max() / counts.min()
        print(f"\nRatio de desbalance mayoría/minoría: {ratio:,.0f}:1 "
              f"({majority} vs {minority}).")
        print("Justificación: con este ratio, accuracy es una métrica engañosa. El "
              "proyecto debe reportar F1/AUC por clase (sección J) y considerar "
              "class_weight, SMOTE controlado o umbral ajustado — nunca oversampling "
              "ciego que duplique masivamente clases con solo 3-8 instancias (u2r).")
    return result

In [ ]:
report["imbalance"] = check_imbalance(df)

### 15. Gaps temporales

In [ ]:
def check_temporal_gaps() -> dict:
    section("15. Gaps temporales")
    print("No aplica de forma directa: no existe timestamp absoluto (ver sección 7). "
          "El orden de filas se usará como proxy temporal para la simulación de "
          "producción (sección P), asumiendo que el archivo preserva el orden de "
          "captura original de la competencia.")
    return {"applicable": False}

In [ ]:
report["temporal_gaps"] = check_temporal_gaps()

### 16. Correlación excesiva

In [ ]:
def check_excessive_correlation(df: pd.DataFrame) -> dict:
    section("16. Correlación excesiva")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [c for c in numeric_cols if c != "is_anomaly"]
    corr = df[numeric_cols].corr().abs()
    pairs = []
    for i, col_i in enumerate(corr.columns):
        for col_j in corr.columns[i + 1:]:
            val = corr.loc[col_i, col_j]
            if val > 0.9:
                pairs.append((col_i, col_j, round(float(val), 3)))
    pairs.sort(key=lambda x: x[2], reverse=True)
    for a, b, v in pairs[:10]:
        print(f"  {a} <-> {b}: r={v}")
    if not pairs:
        print("No se encontraron pares con correlación > 0.9.")
    print("Justificación: pares altamente correlacionados (ej. srv_serror_rate vs "
          "serror_rate) son candidatos a eliminar uno de los dos en Feature Engineering "
          "para reducir redundancia y multicolinealidad en modelos lineales.")
    return {"high_correlation_pairs": pairs}

In [ ]:
report["excessive_correlation"] = check_excessive_correlation(df)

**Nota sobre "proxy" temporal:**

Como ya vimos antes, este dataset no tiene una fecha u hora exacta guardada
para cada conexión. "Proxy" = algo que usamos como sustituto de otra cosa
que no tenemos — en este caso, vamos a usar el orden en que aparecen las
filas en el archivo como si fuera esa fecha/hora, para la simulación de
producción (sección P), asumiendo que el archivo preserva el orden de
captura original de la competencia.

### 17. Anomalías estadísticas (z-score global)

In [ ]:
def check_statistical_anomalies(df: pd.DataFrame) -> dict:
    section("17. Anomalías estadísticas (z-score global)")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [c for c in numeric_cols if c != "is_anomaly"]
    z = (df[numeric_cols] - df[numeric_cols].mean()) / df[numeric_cols].std(ddof=0)
    extreme_rows = (z.abs() > 5).any(axis=1)
    n_extreme = int(extreme_rows.sum())
    pct = round(n_extreme / len(df) * 100, 2)
    print(f"Filas con al menos una columna con |z-score| > 5: {n_extreme:,} ({pct}%).")
    if "is_anomaly" in df.columns:
        overlap = int((extreme_rows & (df["is_anomaly"] == 1)).sum())
        print(f"De esas, {overlap:,} ya están etiquetadas como ataque "
              f"({round(overlap / max(n_extreme, 1) * 100, 1)}% de solapamiento).")
        print("Nota: el solapamiento parcial (no total) confirma que el z-score global "
              "por sí solo NO es un detector de anomalías suficiente para este problema "
              "— hay ataques con features dentro de rango 'normal' y normales con "
              "outliers legítimos (ej. transferencias grandes de archivos).")
    return {"extreme_rows": n_extreme, "extreme_pct": pct}

In [ ]:
report["statistical_anomalies"] = check_statistical_anomalies(df)

**Nota importante sobre el z-score:**

- **"z-score"** = una forma de medir qué tan lejos está un valor del promedio. Un z-score alto significa "este valor es raro comparado con el resto".
- **"Solapamiento"** = cuánto se cruzan o coinciden dos grupos distintos (las filas "raras" según el z-score, y las filas que sabemos que son ataques reales).

El solapamiento parcial (no total) confirma que un método simple de
"buscar valores raros" NO es suficiente para detectar ataques en este
problema. Hay dos tipos de casos que confunden a este método:

1. Ataques reales que NO se ven raros (sus valores están dentro de lo que parecería "normal").
2. Conexiones normales que SÍ se ven raras, pero no son ataques (por ejemplo, alguien transfiriendo un archivo muy grande de forma legítima).

Por eso se necesita un modelo más inteligente (entrenado con IA), no solo
una regla matemática simple que busque valores extremos.

### Guardar el reporte de calidad completo

In [ ]:
output_path = raiz / "reports" / "data_quality_report.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)

print(f"JSON estructurado guardado en: {output_path}")

## Sección H — Análisis Exploratorio de Datos (EDA)

Cada gráfico responde explícitamente la pregunta exigida por el proyecto:
**¿qué decisión de modelado, limpieza, ingeniería de variables o negocio
cambia como consecuencia de este resultado?**

### Preparar librerías de gráficos y carpeta de salida

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)

figs_dir = raiz / "reports" / "figures"
figs_dir.mkdir(parents=True, exist_ok=True)

### EDA 1 — Distribución de categorías de ataque

In [ ]:
def eda_distribucion_clases(df: pd.DataFrame, figs_dir: Path) -> None:
    section("EDA 1 — Distribución de categorías de ataque")
    counts = df["attack_category"].value_counts()

    fig, ax = plt.subplots()
    counts.plot(kind="bar", ax=ax, color="steelblue")
    ax.set_yscale("log")
    ax.set_ylabel("Número de conexiones (escala log)")
    ax.set_title("Distribución de attack_category (escala logarítmica)")
    for i, v in enumerate(counts):
        ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.savefig(figs_dir / "01_distribucion_attack_category.png", dpi=100)
    plt.close(fig)

    print(counts)
    print("\n¿Qué decisión cambia?: Confirma visualmente el desbalance extremo. "
          "Decisión de MODELADO: la métrica de evaluación (sección J de MLflow) debe "
          "ser F1/AUC por clase, nunca accuracy global. Decisión de NEGOCIO: hay que "
          "decidir si el costo de un falso negativo en 'u2r' justifica un umbral de "
          "decisión distinto al de 'dos'.")

In [ ]:
eda_distribucion_clases(df, figs_dir)

### EDA 2 — src_bytes y dst_bytes por categoría de ataque

In [ ]:
def eda_bytes_por_categoria(df: pd.DataFrame, figs_dir: Path) -> None:
    section("EDA 2 — src_bytes y dst_bytes por categoría de ataque")
    orden = df["attack_category"].value_counts().index

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, col in zip(axes, ["src_bytes", "dst_bytes"]):
        data_to_plot = [np.log1p(df.loc[df["attack_category"] == cat, col]) for cat in orden]
        ax.boxplot(data_to_plot, labels=orden, showfliers=False)
        ax.set_title(f"log1p({col}) por categoría")
        ax.set_ylabel(f"log1p({col})")
    plt.tight_layout()
    plt.savefig(figs_dir / "02_bytes_por_categoria.png", dpi=100)
    plt.close(fig)

    medianas = df.groupby("attack_category")[["src_bytes", "dst_bytes"]].median()
    print(medianas)
    print("\n¿Qué decisión cambia?: Confirma que 'src_bytes'/'dst_bytes' son "
          "features discriminativas (ej. 'dos' con dst_bytes casi 0 por ser tráfico "
          "de una sola dirección). Decisión de INGENIERÍA DE VARIABLES: aplicar log1p "
          "antes de modelos lineales/distancia; los modelos de árboles no lo requieren.")

In [ ]:
eda_bytes_por_categoria(df, figs_dir)

### EDA 3 — protocol_type por categoría de ataque

In [ ]:
def eda_protocolo_por_categoria(df: pd.DataFrame, figs_dir: Path) -> None:
    section("EDA 3 — protocol_type por categoría de ataque")
    tabla = pd.crosstab(df["attack_category"], df["protocol_type"], normalize="index") * 100
    print(tabla.round(1))

    fig, ax = plt.subplots()
    tabla.plot(kind="bar", stacked=True, ax=ax)
    ax.set_ylabel("% dentro de cada categoría")
    ax.set_title("Composición de protocol_type dentro de cada attack_category")
    ax.legend(title="protocol_type")
    plt.tight_layout()
    plt.savefig(figs_dir / "03_protocolo_por_categoria.png", dpi=100)
    plt.close(fig)

    print("\n¿Qué decisión cambia?: Si 'dos' se concentra fuertemente en icmp y "
          "'r2l'/'u2r' en tcp, confirma 'protocol_type' como señal fuerte y barata. "
          "Decisión de NEGOCIO: posible pre-filtro rápido por protocolo en el API "
          "antes de correr el modelo completo (reduce latencia, sección O1).")

In [ ]:
eda_protocolo_por_categoria(df, figs_dir)

### EDA 4 — Servicios con mayor tasa de anomalía

In [ ]:
def eda_servicios_riesgosos(df: pd.DataFrame, figs_dir: Path) -> None:
    section("EDA 4 — Servicios con mayor tasa de anomalía")
    tasa = (
        df.groupby("service")["is_anomaly"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "tasa_anomalia", "count": "n_conexiones"})
    )
    tasa_confiable = tasa[tasa["n_conexiones"] >= 50]
    top15 = tasa_confiable.sort_values("tasa_anomalia", ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(10, 6))
    top15["tasa_anomalia"].plot(kind="barh", ax=ax, color="firebrick")
    ax.set_xlabel("Proporción de conexiones anómalas")
    ax.set_title("Top 15 servicios con mayor tasa de anomalía (min. 50 conexiones)")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(figs_dir / "04_servicios_riesgosos.png", dpi=100)
    plt.close(fig)

    print(top15)
    print("\n¿Qué decisión cambia?: Decisión de INGENIERÍA DE VARIABLES: usar target "
          "encoding de 'service' (tasa de riesgo histórica) en vez de one-hot de 66 "
          "categorías. ADVERTENCIA para el informe: si algún servicio sale con tasa "
          "exacta de 100%, es probable que refleje una limitación conocida del "
          "entorno simulado de este dataset (McHugh, 2000) — servicios poco comunes "
          "que solo aparecen en tráfico de probing, no una relación causal real. "
          "Documentar como limitación, no como hallazgo definitivo.")

In [ ]:
eda_servicios_riesgosos(df, figs_dir)

### EDA 5 — Tasas de error (serror_rate) por categoría

In [ ]:
def eda_serror_rate_por_categoria(df: pd.DataFrame, figs_dir: Path) -> None:
    section("EDA 5 — Tasas de error (serror_rate) por categoría")
    orden = df["attack_category"].value_counts().index

    fig, ax = plt.subplots()
    for cat in orden:
        subset = df.loc[df["attack_category"] == cat, "serror_rate"]
        ax.hist(subset, bins=20, alpha=0.5, label=cat, density=True)
    ax.set_xlabel("serror_rate")
    ax.set_ylabel("Densidad")
    ax.set_title("Distribución de serror_rate por categoría de ataque")
    ax.legend()
    plt.tight_layout()
    plt.savefig(figs_dir / "05_serror_rate_por_categoria.png", dpi=100)
    plt.close(fig)

    medias = df.groupby("attack_category")["serror_rate"].mean().sort_values(ascending=False)
    print(medias)
    print("\n¿Qué decisión cambia?: De los 17 pares correlacionados (r>0.9) "
          "identificados en calidad de datos, 'serror_rate', 'srv_serror_rate' y "
          "'dst_host_serror_rate' aportan señal muy similar entre sí. Decisión de "
          "INGENIERÍA DE VARIABLES: conservar solo 'dst_host_serror_rate' (ventana "
          "más estable, 100 conexiones) y descartar las otras dos para reducir "
          "multicolinealidad sin perder señal.")

In [ ]:
eda_serror_rate_por_categoria(df, figs_dir)

### EDA 6 — Mapa de calor de correlación y ranking vs. is_anomaly

In [ ]:
def eda_correlacion_con_target(df: pd.DataFrame, figs_dir: Path) -> dict:
    section("EDA 6 — Mapa de calor de correlación y ranking vs. is_anomaly")
    cols_interes = [
        "duration", "src_bytes", "dst_bytes", "count", "srv_count",
        "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
        "same_srv_rate", "diff_srv_rate", "dst_host_serror_rate",
        "dst_host_same_srv_rate", "is_anomaly",
    ]
    cols_interes = [c for c in cols_interes if c in df.columns]
    corr = df[cols_interes].corr()

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(cols_interes)))
    ax.set_yticks(range(len(cols_interes)))
    ax.set_xticklabels(cols_interes, rotation=90)
    ax.set_yticklabels(cols_interes)
    plt.colorbar(im, ax=ax, label="Correlación")
    ax.set_title("Mapa de calor — features numéricas clave + is_anomaly")
    plt.tight_layout()
    plt.savefig(figs_dir / "06_heatmap_correlacion.png", dpi=100)
    plt.close(fig)

    corr_con_target = corr["is_anomaly"].drop("is_anomaly").sort_values(ascending=False)
    print("Correlación de cada feature con is_anomaly (target):")
    print(corr_con_target)
    print("\n¿Qué decisión cambia?: El set de features prioritario para el modelo "
          "baseline (Run 001 de MLflow, sección J) debe anclarse en las de mayor "
          "correlación absoluta con el target — típicamente 'count'/'srv_count' por "
          "encima de las tasas de error en este dataset. Recordar que 'count' y "
          "'srv_count' pertenecen a las columnas de ventana de 2 segundos marcadas "
          "en Leakage (sección F): al ser de las features MÁS valiosas, es crítico "
          "que el API de inferencia las recalcule de forma idéntica al histórico.")
    return corr_con_target.to_dict()

In [ ]:
corr_con_target = eda_correlacion_con_target(df, figs_dir)

### Resumen — Decisiones que cambian a partir de este EDA

In [ ]:
    section("Resumen — Decisiones que cambian a partir de este EDA")
    resumen = [
        ("1. Distribución de clases", "Métrica = F1/AUC por clase, no accuracy"),
        ("2. Bytes por categoría", "Transformar con log1p, no eliminar la columna"),
        ("3. Protocolo por categoría", "Posible pre-filtro rápido por protocolo en el API"),
        ("4. Servicios riesgosos", "Target encoding de 'service'; documentar limitación del dataset"),
        ("5. serror_rate por categoría", "Conservar 'dst_host_serror_rate', descartar variantes redundantes"),
        ("6. Heatmap + correlación", "Set de features priorizado; 'count'/'srv_count' críticas para el API"),
    ]
    for grafico, decision in resumen:
        print(f"  {grafico}: {decision}")

    print(f"\nFiguras guardadas en: {figs_dir}")
    for f in sorted(figs_dir.glob("*.png")):
        print(f"  - {f.name}")